In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 📘 Explicación del Generador de Metaprompt

Este código automatiza la creación de instrucciones complejas para IAs, orientadas a la escritura técnica **IEEE**. Sus componentes principales son:

1.  **Entrada Dinámica (`input`)**: Captura el tema de investigación del usuario.
2.  **Plantilla de Ingeniería de Prompts (`Template`)**: Define un esquema estricto de tres partes (Estructura JSON, Bibliografía BibTeX 2023-2026 y Mapa de Uso) que fuerza a la IA a ser técnica y organizada.
3.  **Interfaz de Usuario (`ipywidgets`)**: Crea un área de texto para revisar el prompt generado.
4.  **Botón de Copiado Seguro (`JavaScript`)**: Implementa un script personalizado para evadir las restricciones de seguridad de Colab, permitiendo copiar el texto al portapapeles con un solo clic.

**Objetivo:** Obtener un 'plano' detallado del paper antes de empezar a redactar.

In [ ]:
# @title 🛠️ Generador de Metaprompt para Documentación IEEE
# @markdown Ingrese el tema o contenido de la investigación para procesar el metaprompt.

# 1. Captura de parámetros mediante input de texto
contenido_insertado = input("Ingrese el tema de investigación: ")

import json
from string import Template
from ipywidgets import widgets
from IPython.display import display, HTML

# 2. Definición del Metaprompt usando Template
metaprompt_tpl = Template("""Actúa como Editor Jefe de IEEE. Para el tema de investigación **$contenido**, genera un **Desglose Temático Detallado** siguiendo estrictamente este esquema de tres partes:

**PARTE 1: ESTRUCTURA JSON**
Entrega un único bloque de código JSON con la siguiente estructura:
```json
{
  "titulo": "Título técnico en español",
  "folder_name": "Identificador alfanumérico del proyecto bajo la convención snake_case. Sin espacios en blanco. Usar '_' como separador",
  "abstract_preliminar": "Resumen técnico de 150 palabras siguiendo normas IEEE",
  "secciones": [
    {
      "nro": 1,
      "titulo_seccion": "Nombre de la sección",
      "objetivos": ["objetivo 1", "objetivo 2"],
      "subsecciones": ["1.1...", "1.2..."],
      "insumos": ["Tabla 1", "Eq. 1"],
      "llaves_bibtex": ["Key1", "Key2"]
    }
  ]
}
```

**PARTE 2: BLOQUES BIBLIOGRÁFICOS SECCIONALES**
Para cada sección, entrega un bloque de código independiente con las entradas en formato `BibTeX`.
* Fuentes reales, **verificables** (Accesibles desde la web mediante url) y publicadas entre 2023 y 2026.
* La suma total de referencias en TODAS las secciones debe ser ≤ 30.
* Formato limpio, listo para copiar/pegar directamente en `references.bib`.

**PARTE 3: MAPA DE USO DE REFERENCIAS (POR SECCIÓN)**
Para cada sección, genera un bloque JSON independiente que funcione como diccionario asociativo. Debe conectar **textualmente** cada `llave_bibtex` de la PARTE 1 con su justificación y directrices de integración:
```json
{
  "seccion_nro": 1,
  "titulo_seccion": "Nombre exacto de la sección",
  "mapa_uso": {
    "Key1": {
      "razon_seleccion": "Justificación técnica o metodológica de la elección (máx. 1 oración).",
      "guia_redaccion": "Instrucción precisa sobre cómo integrarla (ej: 'Usar en 1.1 para contrastar X vs Y, citando resultados de eficiencia y destacando limitaciones en entornos reales').",
      "subseccion_destino": "1.1"
    },
    "Key2": { "..." : "..." }
  }
}
```

**🔒 INSTRUCCIONES CRÍTICAS:**
1. **Coherencia absoluta:** Las claves en `llaves_bibtex` (PARTE 1), en los bloques `.bib` (PARTE 2) y en `mapa_uso` (PARTE 3) deben coincidir carácter por carácter.
2. **Sin redacción aún:** No generes párrafos, introducciones ni conclusiones. Solo entrega los bloques de código solicitados.
3. **Formato estricto:** Cada parte debe ir en su propio bloque de código. No incluyas texto explicativo, saludos ni comentarios entre bloques.
4. **Enfoque IEEE:** Prioriza referencias de journals/conferencias indexados, métricas cuantitativas y metodologías reproducibles.""")

# 3. Sustitución de parámetros
prompt_final = metaprompt_tpl.substitute(contenido=contenido_insertado)

# 4. Interfaz de salida
print("\n✅ Prompt procesado con éxito.\n")

# Preparamos el string para JS de forma segura
prompt_json_esc = json.dumps(prompt_final)

# Botón para copiar con fallback a textarea para máxima compatibilidad en Colab
boton_copiar_html = HTML(f"""
    <script>
    function copiarAlPortapapeles() {{
        const text = {prompt_json_esc};
        const textArea = document.createElement("textarea");
        textArea.value = text;
        document.body.appendChild(textArea);
        textArea.select();
        try {{
            document.execCommand('copy');
            alert("Prompt copiado al portapapeles");
        }} catch (err) {{
            console.error('Error al copiar: ', err);
        }}
        document.body.removeChild(textArea);
    }}
    </script>
    <button onclick="copiarAlPortapapeles()"
    style="background-color: #4CAF50; color: white; padding: 10px 20px; border: none; border-radius: 4px; cursor: pointer; margin-bottom: 10px;">
    📋 Copiar Prompt al Portapapeles
    </button>
""")

output_area = widgets.Textarea(
    value=prompt_final,
    layout=widgets.Layout(width='98%', height='300px'),
    description='Prompt:',
    disabled=False
)

display(boton_copiar_html)
display(output_area)

Ingrese el tema de investigación: Segmentación Semántica Desastres (Acelera respuesta a emergencias en tiempo real.)

✅ Prompt procesado con éxito.



Textarea(value='Actúa como Editor Jefe de IEEE. Para el tema de investigación **Segmentación Semántica Desastr…

### 📝 Gestor de Plan de Investigación

Este componente proporciona una interfaz interactiva para capturar y persistir la estructura detallada de la investigación. Sus funciones son:

1.  **Captura de Contenido**: Un área de texto de gran capacidad para pegar el plan en formato Markdown.
2.  **Persistencia Directa**: El botón **Guardar** crea automáticamente el directorio necesario y escribe el archivo en `workflow/workflow/research_plan.md`.
3.  **Control de Flujo**: Permite limpiar el área de trabajo rápidamente para nuevas iteraciones.
4.  **Validación**: Informa al usuario sobre el éxito de la operación y el tamaño del archivo guardado.

**Importancia:** Este archivo es el insumo principal para el procesador que genera el entorno LaTeX y los prompts de escritura.

In [1]:
import os
from ipywidgets import widgets
from IPython.display import display

# Configuración de ruta
file_path = 'workflow/workflow/research_plan.md'

# 1. Crear widgets
text_area = widgets.Textarea(
    placeholder='Pegue aquí el contenido largo en Markdown...',
    description='Contenido:',
    layout=widgets.Layout(width='98%', height='400px')
)

save_button = widgets.Button(
    description='💾 Guardar Plan de Investigación',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

clear_button = widgets.Button(
    description='🗑️ Limpiar Área',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

# 2. Funciones de control
def on_save_clicked(b):
    with output:
        output.clear_output()
        try:
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text_area.value)
            print(f"✅ Archivo guardado exitosamente en: {file_path}")
            print(f"📏 Tamaño: {len(text_area.value)} caracteres.")
        except Exception as e:
            print(f"❌ Error al guardar: {str(e)}")

def on_clear_clicked(b):
    text_area.value = ''
    with output:
        output.clear_output()
        print("🧹 Área de texto limpiada.")

save_button.on_click(on_save_clicked)
clear_button.on_click(on_clear_clicked)

# 3. Mostrar interfaz
print("📝 Gestor de Plan de Investigación")
display(text_area)
display(widgets.HBox([save_button, clear_button]))
display(output)

📝 Gestor de Plan de Investigación


Textarea(value='', description='Contenido:', layout=Layout(height='400px', width='98%'), placeholder='Pegue aq…

Output()

In [ ]:
#rm -r workflow/workflow/outputs/

### ⚙️ Procesador de Documentos (SRA 5.2)

Este script es el motor de automatización que transforma el plan de investigación en un entorno de trabajo LaTeX profesional. Sus funciones principales son:

1.  **Extracción de Bloques**: Utiliza Regex para identificar y separar la estructura JSON y los bloques BibTeX dentro del archivo Markdown.
2.  **Generación de Estructura IEEE**: Crea automáticamente el archivo `main.tex` con el preámbulo oficial de IEEE, metadatos del autor y configuración de `biblatex`.
3.  **Modularización**: Genera archivos independientes (`section_*.tex`) para cada parte del documento, facilitando la edición por separado.
4.  **Ingeniería de Prompts por Sección**: Crea archivos de texto (`prompt_section_*.txt`) con instrucciones específicas para alimentar a una IA, incluyendo:
    *   Objetivos técnicos de la sección.
    *   Citas bibliográficas obligatorias.
    *   Mapa de uso de referencias.
    *   Restricciones de formato para evitar errores de compilación.
5.  **Gestión Bibliográfica**: Consolida todas las fuentes en un archivo `references.bib` listo para ser procesado por Biber.

**Resultado:** Un entorno de desarrollo listo para cargar en Overleaf o compilar localmente con redacción asistida por IA.

In [2]:
import json
import re
import os
import sys


def slugify(text):
    """Convierte el nombre del tema en un nombre de carpeta válido."""
    if not text:
        return "investigacion_nueva"
    text = text.lower().replace(" ", "_")
    return re.sub(r'(?u)[^-\w.]', '', text)


def extract_document_blocks(content):
    """
    Extrae todos los bloques de contenido relevantes del archivo.
    Retorna: (json_estructura_principal, dict_mapas_por_seccion, lista_bibtex)
    """
    json_blocks = re.findall(r'```json\s*(\{.*?\})\s*```', content, re.DOTALL)
    bib_blocks = re.findall(r'```bib(?:tex)?\s*(.*?)\s*```', content, re.DOTALL)

    if not json_blocks:
        return None, {}, bib_blocks

    estructura_principal = None
    mapas_por_seccion = {}

    for block in json_blocks:
        try:
            data = json.loads(block)
            if "titulo" in data and "secciones" in data:
                estructura_principal = data
            elif "mapa_uso" in data or "seccion_nro" in data:
                nro_seccion = data.get("seccion_nro", data.get("seccion", 0))
                mapas_por_seccion[nro_seccion] = data
        except json.JSONDecodeError:
            continue

    return estructura_principal, mapas_por_seccion, bib_blocks


def create_output_directory(input_file, project_name):
    input_dir = os.path.dirname(os.path.abspath(input_file))
    folder_name = slugify(project_name)
    output_path = os.path.join(input_dir, folder_name)

    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Carpeta creada exitosamente en: {output_path}")
    else:
        print(f"Usando carpeta existente: {output_path}")

    return output_path


def load_image_manifest(output_path):
    manifest_path = os.path.join(output_path, 'image_manifest.json')
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, 'r', encoding='utf-8') as mf:
                return json.load(mf)
        except json.JSONDecodeError:
            return {}
    return {}


def save_image_manifest(output_path, manifest_data):
    manifest_path = os.path.join(output_path, 'image_manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as mf:
        json.dump(manifest_data, mf, indent=4, ensure_ascii=False)


def write_section_tex_files(data, output_path):
    sections = data.get('secciones', [])
    created = []

    for sec in sections:
        n = sec.get('nro', 0)
        sec_title = sec.get('titulo_seccion', f'Seccion {n}')
        filename = os.path.join(output_path, f'section_{n}.tex')
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"% Contenido para la sección: {sec_title}\n")
            f.write(f"\\section{{{sec_title}}}\n")
        created.append(filename)

    print(f"✅ Generados {len(created)} archivos section_*.tex")
    return created


def build_citation_instructions(sec, mapas_referencias):
    n = sec.get('nro', 0)
    mapa_uso = mapas_referencias.get(n, {})
    mapa_detalle = mapa_uso.get('mapa_uso', {})
    instrucciones = []

    if mapa_detalle:
        instrucciones.append("\n📚 MAPA DE USO DE REFERENCIAS (OBLIGATORIO):")
        for key, info in mapa_detalle.items():
            razon = info.get('razon_seleccion', 'Sin justificación')
            guia = info.get('guia_redaccion', 'Integrar de forma natural')
            subseccion = info.get('subseccion_destino', 'Cualquiera')
            instrucciones.append(f"\n• **{key}** (Subsección {subseccion}):")
            instrucciones.append(f"  - Razón: {razon}")
            instrucciones.append(f"  - Instrucción: {guia}")

    return '\n'.join(instrucciones)


def write_prompt_section_files(data, mapas_referencias, output_path, project_name):
    sections = data.get('secciones', [])
    manifest_data = load_image_manifest(output_path)
    created = []

    for sec in sections:
        n = sec.get('nro', 0)
        sec_title = sec.get('titulo_seccion', f'Seccion {n}')
        filename = os.path.join(output_path, f'prompt_section_{n}.txt')
        instrucciones_citacion_text = build_citation_instructions(sec, mapas_referencias)

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"=== SPRINT DE ESCRITURA: SECCIÓN {n} ===\n\n")
            f.write(f"PAPER: {data.get('titulo', project_name)}\n")
            f.write(f"SECCIÓN: {sec_title}\n\n")
            f.write("--- CONTEXTO Y OBJETIVOS ---\n")
            f.write(f"OBJETIVOS: {', '.join(sec.get('objetivos', []))}\n")
            f.write(f"SUBSECCIONES REQUERIDAS: {', '.join(sec.get('subsecciones', []))}\n")
            f.write(f"INSUMOS (Tablas/Ecuaciones/Figuras): {', '.join(sec.get('insumos', []))}\n\n")
            f.write("--- CITAS BIBLIOGRÁFICAS OBLIGATORIAS ---\n")
            f.write(f"Claves BibTeX: {', '.join(sec.get('llaves_bibtex', []))}\n")

            if instrucciones_citacion_text:
                f.write(instrucciones_citacion_text)
            else:
                f.write("\n⚠️ ADVERTENCIA: No se encontró mapa de uso para esta sección. Integra las citas de forma coherente.\n")

            f.write("\n\n--- ⛔ RESTRICCIONES MODULARES (CRÍTICO) ---\n")
            f.write("Este fragmento se insertará vía \\input{section_N.tex} en un main.tex YA CONFIGURADO.\n")
            f.write("PROHIBIDO INCLUIR:\n")
            f.write("  • \\documentclass, \\begin{document}, \\end{document}\n")
            f.write("  • \\usepackage{}, \\addbibresource{}, \\bibliographystyle{}, \\bibliography{}\n")
            f.write("  • \\begin{filecontents}{...} ... \\end{filecontents}\n")
            f.write("  • Preambulo, metadatos, autor, título o configuración de bibliografía.\n")
            f.write("  • Comentarios explicativos fuera de LaTeX.\n\n")
            f.write("--- ✅ INSTRUCCIONES DE SALIDA ---\n")
            f.write("\n\n--- ⚙️ FORMATO DE SALIDA ESTRICTO (OBLIGATORIO) ---\n")
            f.write("Tu respuesta debe contener EXACTAMENTE dos bloques de código Markdown. Nada más.\n\n")
            f.write("BLOQUE 1 (Contenido LaTeX):\n")
            f.write("```latex\n")
            f.write("\\section{Nombre Sección}\n")
            f.write("\\subsection{Subsección 1.1}\n")
            f.write("Texto académico con \\cite{Key} ...\n")
            f.write("\\begin{figure}[h]\\centering\\includegraphics[width=\\linewidth]{fig.png}\\caption{Desc}\\label{fig:1}\\end{figure}\n")
            f.write("```\n\n")
            f.write("BLOQUE 2 (Prompts de Imagen):\n")
            f.write("```json\n")
            f.write("{\n")
            f.write('  "fig1.png": "2D technical vector diagram..."\n')
            f.write("}\n")
            f.write("```\n\n")
            f.write("1. EXTENSIÓN: 600-800 palabras.\n")
            f.write("2. ESTRUCTURA: Solo \\section{} y \\subsection{} según lo listado.\n")
            f.write("3. CITAS: Usa \\cite{clave} en línea. NUNCA menciones una referencia sin citarla.\n")
            f.write("4. FIGURAS/TABLAS: Usa entornos figure/table estándar de IEEE.\n")
            f.write("5. AL FINAL: Un único bloque ```json con prompts para DALL-E 3.\n\n")
            f.write("FORMATO DE FIGURAS:\n")
            f.write("\\begin{figure}[h]\n\\centering\n\\includegraphics[width=\\linewidth]{nombre_archivo.png}\n\\caption{Descripción técnica}\n\\label{fig:nombre_archivo}\n\\end{figure}\n\n")
            f.write("REGLAS PARA PROMPTS DE IMAGEN:\n")
            f.write("- Estilo: '2D technical vector diagram, engineering schematic, flat design'\n")
            f.write("- Fondo: Blanco puro, sin texto interno, sin perspectiva 3D\n")
            f.write("- Colores: Azul cobalto (#0047AB), Gris técnico (#4A4A4A), Negro\n")
            f.write("- Evita: Sombras, gradientes, elementos decorativos\n")

        created.append(filename)

        for insumo in sec.get('insumos', []):
            if 'Fig' in insumo or 'Imagen' in insumo or 'Figure' in insumo:
                img_key = slugify(insumo) + '.png'
                if img_key not in manifest_data:
                    manifest_data[img_key] = {
                        'seccion': n,
                        'descripcion_original': insumo,
                        'prompt_ia': 'Pendiente de generación por el Sprint de Escritura'
                    }

    save_image_manifest(output_path, manifest_data)
    print(f"✅ Generados {len(created)} archivos prompt_section_*.txt")
    return created


def write_register_prompt_file(data, output_path, project_name):
    filename = os.path.join(output_path, 'prompt_register.txt')
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f'Necesito ingresar en el sistema de gestión documental de la Agencia los datos de la investigación: **{data.get("titulo", project_name)}**.\n\n')
        f.write('Campos requeridos:\n')
        f.write('* Title\n')
        f.write('* Description\n')
        f.write('* General Objective\n')
        f.write('* Specific Objectives\n')
        f.write('* Justification\n')
        f.write('* Methodology\n')
        f.write('* Scope (máximo 200 caracteres)\n')
        f.write('* Activities\n')
        f.write('* Resources\n')
        f.write('* Limitations\n\n')
        f.write('Proporciona los valores en español basándote en el abstract y la estructura del paper.')
    print(f"✅ Generado archivo prompt_register.txt")
    return filename


def generate_section_files(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_section_tex_files(data, output_path)


def generate_prompt_files(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_prompt_section_files(data, mapas_referencias, output_path, project_name)


def generate_register_prompt_file(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_register_prompt_file(data, output_path, project_name)


def generate_research_files(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    content = content.replace('\u00a0', ' ')
    data, mapas_referencias, bib_blocks = extract_document_blocks(content)

    if not data:
        print("Error: No se encontró el JSON de estructura principal (debe contener 'titulo' y 'secciones').")
        return

    if not bib_blocks:
        print("Error: No se encontraron bloques BibTeX en el archivo.")
        return

    if not project_name:
        project_name = data.get('titulo', 'investigacion_nueva')

    output_path = create_output_directory(input_file, project_name)
    write_section_tex_files(data, output_path)
    write_prompt_section_files(data, mapas_referencias, output_path, project_name)
    write_register_prompt_file(data, output_path, project_name)

    full_bib_content = '\n\n'.join([b.strip() for b in bib_blocks])
    with open(os.path.join(output_path, 'references.bib'), 'w', encoding='utf-8') as f:
        f.write(full_bib_content)

    abstract_content = data.get('abstract_preliminar', 'Abstract no disponible.')
    with open(os.path.join(output_path, 'abstract.tex'), 'w', encoding='utf-8') as f:
        f.write('\\begin{abstract}\n' + abstract_content + '\n\\end{abstract}')

    sections = data.get('secciones', [])
    main_tex = r"""\documentclass[10pt, journal, final, twocolumn, letterpaper]{IEEEtran}

% --- PREÁMBULO DE PAQUETES ---
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage{amsmath, amssymb, amsfonts, amsthm}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{array}
\usepackage{url}
\usepackage{hyperref}
\usepackage{color, xcolor}
\usepackage[backend=biber, style=ieee, natbib=true]{biblatex}

% --- CONFIGURACIÓN DE BIBLIOGRAFÍA ---
\addbibresource{references.bib}

% --- METADATOS DEL DOCUMENTO ---
\title{""" + data.get('titulo', project_name) + r"""

\author{\IEEEauthorblockN{Lic. Héctor Martínez\\}
\IEEEauthorblockA{Unidad de Telecomunicaciones\\
Agencia Bolivariana para Actividades Espaciales\\
Email: hmartinez@abae.gob.ve}}

\begin{document}
\maketitle

\input{abstract.tex}

"""

    for sec in sections:
        n = sec.get('nro', 0)
        main_tex += f"\\input{{section_{n}.tex}}\n"

    main_tex += "\n% --- BIBLIOGRAFÍA ---\n%\\nocite{*}\n" + "\\printbibliography" + "\n\\end{document}"
    with open(os.path.join(output_path, 'main.tex'), 'w', encoding='utf-8') as f:
        f.write(main_tex)

    print(f"✅ Éxito. Archivos generados en: {project_name}")
    print(f"📊 Mapas de uso detectados: {len(mapas_referencias)} secciones")
    if mapas_referencias:
        print(f"   Secciones con mapa: {sorted(mapas_referencias.keys())}")


def print_usage():
    print('Uso: python sra_5.2.py [modo] [archivo_entrada] [nombre_proyecto]')
    print('Modos disponibles: all, sections, prompts, register')
    print('Ejemplo: python sra_5.2.py sections research_plan.md')


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else 'all'
    input_file = sys.argv[2] if len(sys.argv) > 2 else 'workflow/workflow/research_plan.md'
    project_name = sys.argv[3] if len(sys.argv) > 3 else None

    if mode == 'all':
        generate_research_files(input_file, project_name)
    elif mode == 'sections':
        generate_section_files(input_file, project_name)
    elif mode == 'prompts':
        generate_prompt_files(input_file, project_name)
    elif mode == 'register':
        generate_register_prompt_file(input_file, project_name)
    else:
        print_usage()


Uso: python sra_5.2.py [modo] [archivo_entrada] [nombre_proyecto]
Modos disponibles: all, sections, prompts, register
Ejemplo: python sra_5.2.py sections research_plan.md


In [3]:
import sys

# Define the input file and mode
input_file = 'workflow/workflow/research_plan.md'
mode = 'all'

# Check if the research plan exists before running
import os
if os.path.exists(input_file):
    print(f"🚀 Starting project generation for: {input_file}\n")
    # We call the main function from the previous cell directly for better integration in the notebook
    generate_research_files(input_file, 'outputs')
else:
    print(f"❌ Error: The file {input_file} was not found. Please ensure you saved the research plan first.")

🚀 Starting project generation for: workflow/workflow/research_plan.md

Carpeta creada exitosamente en: /content/workflow/workflow/outputs
✅ Generados 5 archivos section_*.tex
✅ Generados 5 archivos prompt_section_*.txt
✅ Generado archivo prompt_register.txt
✅ Éxito. Archivos generados en: outputs
📊 Mapas de uso detectados: 5 secciones
   Secciones con mapa: [1, 2, 3, 4, 5]


In [ ]:
import os
import shutil
from google.colab import files

# Direct path to the generated outputs
folder_to_zip = 'workflow/workflow/outputs/'
zip_filename = 'outputs.zip'

if os.path.exists(folder_to_zip):
    # Create zip from the outputs directory
    shutil.make_archive('outputs', 'zip', folder_to_zip)

    print(f"✅ Folder '{folder_to_zip}' compressed successfully.")
    files.download('outputs.zip')
else:
    print(f"❌ Error: Folder '{folder_to_zip}' not found. Please run the generation cell first.")

✅ Folder 'workflow/workflow/outputs/' compressed successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 🚀 Visualizador de Prompts por Sección

Este componente final permite interactuar con los resultados del procesador sin salir de la interfaz de Colab. Sus funciones clave son:

1.  **Exploración Granular**: Permite cargar individualmente el prompt de cualquier sección mediante su número identificador.
2.  **Copiado Seguro e Instantáneo**: Integra el mismo motor de JavaScript desarrollado para el Metaprompt, asegurando que el contenido técnico (usualmente muy largo) se copie correctamente al portapapeles.
3.  **Previsualización**: Ofrece un área de texto para validar el contenido antes de usarlo en el chat de redacción asistida.

**Flujo de Trabajo:** Ingrese el número de sección -> Cargar -> Copiar -> Pegar en su IA de redacción preferida.

In [6]:
import os
import json
from ipywidgets import widgets
from IPython.display import display, HTML

# Configuration
outputs_path = 'workflow/workflow/outputs/'

def get_prompt_content(section_number):
    filename = os.path.join(outputs_path, f'prompt_section_{section_number}.txt')
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            return f.read()
    return f"❌ Error: No se encontró el prompt para la sección {section_number}."

# 1. Widgets
section_input = widgets.IntText(value=1, description='Sección Nro:', layout=widgets.Layout(width='200px'))
load_button = widgets.Button(description='🔍 Cargar Prompt', button_style='info')
output_display = widgets.Output()

def on_load_clicked(b):
    with output_display:
        output_display.clear_output()
        prompt_text = get_prompt_content(section_input.value)

        if "❌" in prompt_text:
            print(prompt_text)
            return

        # Prepare JS safe string
        prompt_json_esc = json.dumps(prompt_text)

        # 2. UI with Copy Button
        boton_copiar_html = HTML(f"""
            <script>
            function copiarPromptSeccion() {{
                const text = {prompt_json_esc};
                const textArea = document.createElement("textarea");
                textArea.value = text;
                document.body.appendChild(textArea);
                textArea.select();
                try {{
                    document.execCommand('copy');
                    alert("Prompt de Sección " + {section_input.value} + " copiado!");
                }} catch (err) {{
                    console.error('Error al copiar: ', err);
                }}
                document.body.removeChild(textArea);
            }}
            </script>
            <button onclick="copiarPromptSeccion()"
            style="background-color: #007bff; color: white; padding: 10px 20px; border: none; border-radius: 4px; cursor: pointer; margin-bottom: 10px;">
            📋 Copiar Prompt al Portapapeles
            </button>
        """)

        prompt_area = widgets.Textarea(
            value=prompt_text,
            layout=widgets.Layout(width='98%', height='400px'),
            disabled=False
        )

        display(boton_copiar_html)
        display(prompt_area)

load_button.on_click(on_load_clicked)

# 3. Layout
print("🚀 Visualizador de Prompts por Sección")
display(widgets.HBox([section_input, load_button]))
display(output_display)

🚀 Visualizador de Prompts por Sección


Output()

In [11]:
import os
import json
import re
from ipywidgets import widgets
from IPython.display import display, HTML

# Esta celda usa las funciones definidas previamente:
# extract_document_blocks, build_citation_instructions, slugify

input_plan_path = 'workflow/workflow/research_plan.md'

def generate_prompt_on_the_fly(section_number):
    if not os.path.exists(input_plan_path):
        return f"❌ Error: No se encontró el archivo {input_plan_path}."

    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    data, mapas_referencias, _ = extract_document_blocks(content)

    if not data:
        return "❌ Error: No se pudo extraer la estructura del plan."

    sections = data.get('secciones', [])
    sec = next((s for s in sections if s.get('nro') == section_number), None)

    if not sec:
        return f"❌ Error: No se encontró la sección {section_number} en el plan."

    # Reconstrucción del Prompt siguiendo la lógica completa de write_prompt_section_files
    sec_title = sec.get('titulo_seccion', f'Sección {section_number}')
    project_title = data.get('titulo', 'Investigación')
    instrucciones_citacion = build_citation_instructions(sec, mapas_referencias)

    prompt = f"=== SPRINT DE ESCRITURA: SECCIÓN {section_number} ===\n\n"
    prompt += f"PAPER: {project_title}\n"
    prompt += f"SECCIÓN: {sec_title}\n\n"
    prompt += "--- CONTEXTO Y OBJETIVOS ---\n"
    prompt += f"OBJETIVOS: {', '.join(sec.get('objetivos', []))}\n"
    prompt += f"SUBSECCIONES REQUERIDAS: {', '.join(sec.get('subsecciones', []))}\n"
    prompt += f"INSUMOS (Tablas/Ecuaciones/Figuras): {', '.join(sec.get('insumos', []))}\n\n"
    prompt += "--- CITAS BIBLIOGRÁFICAS OBLIGATORIAS ---\n"
    prompt += f"Claves BibTeX: {', '.join(sec.get('llaves_bibtex', []))}\n"

    if instrucciones_citacion:
        prompt += instrucciones_citacion
    else:
        prompt += "\n⚠️ ADVERTENCIA: No se encontró mapa de uso para esta sección. Integra las citas de forma coherente.\n"

    prompt += "\n\n--- ⛔ RESTRICCIONES MODULARES (CRÍTICO) ---\n"
    prompt += "Este fragmento se insertará vía \\input{section_N.tex} en un main.tex YA CONFIGURADO.\n"
    prompt += "PROHIBIDO INCLUIR:\n"
    prompt += "  • \\documentclass, \\begin{document}, \\end{document}\n"
    prompt += "  • \\usepackage{}, \\addbibresource{}, \\bibliographystyle{}, \\bibliography{}\n"
    prompt += "  • \\begin{filecontents}{...} ... \\end{filecontents}\n"
    prompt += "  • Preambulo, metadatos, autor, título o configuración de bibliografía.\n"
    prompt += "  • Comentarios explicativos fuera de LaTeX.\n\n"
    prompt += "--- ✅ INSTRUCCIONES DE SALIDA ---\n\n\n"
    prompt += "--- ⚙️ FORMATO DE SALIDA ESTRICTO (OBLIGATORIO) ---\n"
    prompt += "Tu respuesta debe contener EXACTAMENTE dos bloques de código Markdown. Nada más.\n\n"
    prompt += "BLOQUE 1 (Contenido LaTeX):\n"
    prompt += "```latex\n"
    prompt += "\\section{Nombre Sección}\n"
    prompt += "\\subsection{Subsección 1.1}\n"
    prompt += "Texto académico con \\cite{Key} ...\n"
    prompt += "\\begin{figure}[h]\\centering\\includegraphics[width=\\linewidth]{fig.png}\\caption{Desc}\\label{fig:1}\\end{figure}\n"
    prompt += "```\n\n"
    prompt += "BLOQUE 2 (Prompts de Imagen):\n"
    prompt += "```json\n"
    prompt += "{\n"
    prompt += '  "fig1.png": "2D technical vector diagram..."\n'
    prompt += "}\n"
    prompt += "```\n\n"
    prompt += "1. EXTENSIÓN: 600-800 palabras.\n"
    prompt += "2. ESTRUCTURA: Solo \\section{} y \\subsection{} según lo listado.\n"
    prompt += "3. CITAS: Usa \\cite{clave} en línea. NUNCA menciones una referencia sin citarla.\n"
    prompt += "4. FIGURAS/TABLAS: Usa entornos figure/table estándar de IEEE.\n"
    prompt += "5. AL FINAL: Un único bloque ```json con prompts para DALL-E 3.\n\n"
    prompt += "FORMATO DE FIGURAS:\n"
    prompt += "\\begin{figure}[h]\n\\centering\n\\includegraphics[width=\\linewidth]{nombre_archivo.png}\n\\caption{Descripción técnica}\n\\label{fig:nombre_archivo}\n\\end{figure}\n\n"
    prompt += "REGLAS PARA PROMPTS DE IMAGEN:\n"
    prompt += "- Estilo: '2D technical vector diagram, engineering schematic, flat design'\n"
    prompt += "- Fondo: Blanco puro, sin texto interno, sin perspectiva 3D\n"
    prompt += "- Colores: Azul cobalto (#0047AB), Gris técnico (#4A4A4A), Negro\n"
    prompt += "- Evita: Sombras, gradientes, elementos decorativos\n"

    return prompt

# Interfaz
section_input = widgets.IntText(value=1, description='Sección:', layout=widgets.Layout(width='150px'))
btn_generate = widgets.Button(description='⚡ Generar desde MD', button_style='primary')
out_area = widgets.Output()

def on_gen_clicked(b):
    with out_area:
        out_area.clear_output()
        p_content = generate_prompt_on_the_fly(section_input.value)

        if "❌" in p_content:
            print(p_content)
            return

        p_esc = json.dumps(p_content)
        copy_btn = HTML(f"""
            <script>
            function copyDynamic() {{
                const t = {p_esc};
                const el = document.createElement('textarea'); el.value = t;
                document.body.appendChild(el); el.select();
                document.execCommand('copy'); document.body.removeChild(el);
                alert('Prompt generado y copiado');
            }}
            </script>
            <button onclick="copyDynamic()" style="background:#28a745;color:white;padding:8px;border:none;border-radius:4px;cursor:pointer;">📋 Copiar Prompt</button>
        """)

        display(copy_btn)
        display(widgets.Textarea(value=p_content, layout=widgets.Layout(width='98%', height='350px')))

btn_generate.on_click(on_gen_clicked)
display(widgets.HBox([section_input, btn_generate]), out_area)

Output()

In [12]:
import os
import json
from ipywidgets import widgets
from IPython.display import display, HTML

# Esta celda usa extract_document_blocks definida previamente
input_plan_path = 'workflow/workflow/research_plan.md'

def generate_register_prompt_on_the_fly():
    if not os.path.exists(input_plan_path):
        return f"❌ Error: No se encontró el archivo {input_plan_path}."

    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    data, _, _ = extract_document_blocks(content)

    if not data:
        return "❌ Error: No se pudo extraer la estructura del plan para el registro."

    project_title = data.get('titulo', 'Investigación Nueva')

    prompt = f'Necesito ingresar en el sistema de gestión documental de la Agencia los datos de la investigación: **{project_title}**.\n\n'
    prompt += 'Campos requeridos:\n'
    prompt += '* Title\n'
    prompt += '* Description\n'
    prompt += '* General Objective\n'
    prompt += '* Specific Objectives\n'
    prompt += '* Justification\n'
    prompt += '* Methodology\n'
    prompt += '* Scope (máximo 200 caracteres)\n'
    prompt += '* Activities\n'
    prompt += '* Resources\n'
    prompt += '* Limitations\n\n'
    prompt += 'Proporciona los valores en español basándote en el abstract y la estructura del paper.'

    return prompt

# Interfaz
btn_reg = widgets.Button(description='📝 Generar Prompt Registro', button_style='warning', layout=widgets.Layout(width='250px'))
reg_out = widgets.Output()

def on_reg_clicked(b):
    with reg_out:
        reg_out.clear_output()
        r_content = generate_register_prompt_on_the_fly()

        if "❌" in r_content:
            print(r_content)
            return

        r_esc = json.dumps(r_content)
        copy_btn_reg = HTML(f"""
            <script>
            function copyRegister() {{
                const t = {r_esc};
                const el = document.createElement('textarea'); el.value = t;
                document.body.appendChild(el); el.select();
                document.execCommand('copy'); document.body.removeChild(el);
                alert('Prompt de Registro copiado');
            }}
            </script>
            <button onclick="copyRegister()" style="background:#ff9800;color:white;padding:8px;border:none;border-radius:4px;cursor:pointer;margin-bottom:10px;">📋 Copiar Prompt de Registro</button>
        """)

        display(copy_btn_reg)
        display(widgets.Textarea(value=r_content, layout=widgets.Layout(width='98%', height='300px')))

btn_reg.on_click(on_reg_clicked)
display(btn_reg, reg_out)

Button(button_style='warning', description='📝 Generar Prompt Registro', layout=Layout(width='250px'), style=Bu…

Output()

In [ ]:
import os
from ipywidgets import widgets
from IPython.display import display

# Configuración de ruta
output_file = 'workflow/workflow/section.md'

# 1. Crear widgets
print("📝 Capturador de Redacción (Guardar en section.md)")
text_capture = widgets.Textarea(
    placeholder='Pegue aquí el contenido generado por la IA para la sección...',
    description='Contenido:',
    layout=widgets.Layout(width='98%', height='400px')
)

save_btn = widgets.Button(
    description='💾 Guardar en section.md',
    button_style='primary',
    layout=widgets.Layout(width='250px')
)

status_out = widgets.Output()

# 2. Función de guardado
def save_section_content(b):
    with status_out:
        status_out.clear_output()
        try:
            # Asegurar que el directorio existe
            os.makedirs(os.path.dirname(output_file), exist_ok=True)

            # Escribir el archivo
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(text_capture.value)

            print(f"✅ Contenido guardado exitosamente en: {output_file}")
            print(f"📊 Total: {len(text_capture.value)} caracteres.")
        except Exception as e:
            print(f"❌ Error al guardar el archivo: {str(e)}")

save_btn.on_click(save_section_content)

# 3. Mostrar interfaz
display(text_capture)
display(save_btn)
display(status_out)

📝 Capturador de Redacción (Guardar en section.md)


Textarea(value='', description='Contenido:', layout=Layout(height='400px', width='98%'), placeholder='Pegue aq…

Button(button_style='primary', description='💾 Guardar en section.md', layout=Layout(width='250px'), style=Butt…

Output()